## Cloud cover and boundary layer height: ICON experiments

Compares ICON-CH1 KENDA assimilation experiments against the
NWCSAF satellite cloud mask, and puts their boundary layer heights side by side,
for every hourly analysis.

### Inputs

| Source | Field | Grid |
|---|---|---|
| `FG25/det/lff*` first guess (+1 h) | `CLCT` total cloud cover (%) | ICON native, unstructured |
| `hpbl_test/hpbl_*.grb` | `HPBL` boundary layer height (m) | ICON native, unstructured |
| `MSG_nwcsaf_cosmo1eqc3km_*.nc` | `cma` binary cloud mask | regular ~3 km equirectangular |

### Method

1. **Case discovery.** An hour is processed only if all three inputs (NWCSAF, 801,
   802) exist for that valid time; otherwise it is skipped. `MAX_FRAMES` and
   `FRAME_STRIDE` subsample the resulting list for quick experimentation.

2. **Regridding.** NWCSAF is mapped onto the ICON cells by
   nearest neighbour using a KD-tree. Neighbours further than `MAX_NN_DIST_DEG` (~5 km) are set to `NaN`,
   which blanks ICON cells outside satellite coverage. Interpolating the satellite
   onto the model — rather than the reverse — avoids smoothing away model detail.

3. **Binary agreement.** Each model cell is classified cloudy where
   `CLCT > CLCT_CLOUDY_THRESHOLD`, giving a 0/1 field comparable to the satellite
   mask. Their difference takes three values: `+1` model cloudy where the satellite
   is clear, `−1` model clear where the satellite sees cloud, `0` agreement. The
   fractions of each are reported in the panel legends.

4. **Experiment differences.** `801 − 802` is taken on the **raw** `CLCT` field rather
   than the binary masks, so the magnitude of a disagreement is preserved. `HPBL` is
   differenced directly.

### Caveats

- `CLCT_CLOUDY_THRESHOLD = 0` treats *any* non-zero cloud cover as cloudy, which is
  permissive — ICON often carries small non-zero values over wide areas. Results are
  sensitive to this choice and it is worth re-running at 5 % or 10 %.
- Comparison is against +1 h first-guess fields, not free forecasts.

In [2]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

SAVE_FIGURES = True
SHOW_FIGURES = False
PARALLEL = True        # only takes effect when SAVE_FIGURES=True
MAX_WORKERS = 8

# Cap how many frames get processed. Set MAX_FRAMES to None to run the whole range.
MAX_FRAMES = None
FRAME_STRIDE = 1       # e.g. 3 to take every third hour instead of consecutive

import numpy as np
import xarray as xr
import matplotlib

if SAVE_FIGURES:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

from datetime import datetime, timedelta
from scipy.spatial import cKDTree

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import earthkit.data as ekd

# ==========================================================
# SETTINGS
# ==========================================================

SEARCH_START = datetime(2025, 10, 4, 0)
SEARCH_END = datetime(2025, 10, 10, 23)

NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
NWCSAF_NAME_FMT = "MSG_nwcsaf_cosmo1eqc3km_{:%Y%m%d%H%M}.nc"

ICON_DIRS = {
    "801": "/store_new/mch/msopr/jdelbeke/ICON_TST/801/FG25/det",
    "802": "/store_new/mch/msopr/jdelbeke/ICON_TST/802/FG25/det",
}
ICON_NAME_FMT = "lff{:%Y%m%d%H}"

HPBL_DIR = "/scratch/mch/jdelbeke/hpbl_test"
HPBL_NAME_FMT = "hpbl_{exp}_{time:%Y%m%d%H}.grb"

# a model cell counts as cloudy above this total cloud cover (%)
CLCT_CLOUDY_THRESHOLD = 50

# reject nearest neighbours further away than this (degrees, ~5 km)
MAX_NN_DIST_DEG = 0.05

# 801 - 802 raw cloud cover difference; tighten to 50 or 25 for more contrast
CLCT_DIFF_CMAP = "RdBu_r"
CLCT_DIFF_LIM = 100          # +/- %

HPBL_CMAP = "viridis"
HPBL_VMIN = 0
HPBL_VMAX = 2000

HPBL_DIFF_CMAP = "RdBu_r"
HPBL_DIFF_LIM = 500          # +/- m

MAP_EXTENT = [4, 12, 45, 49]

DOT_SIZE = 2
TITLE_SIZE = 11
TICK_SIZE = 7
LEGEND_SIZE = 7

OUTPUT_DIR = os.path.expanduser("~/CloudAgreement_HPBL")
if SAVE_FIGURES:
    os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================================
# FIND FILES
# ==========================================================

def find_nwcsaf_file(valid_time):
    path = os.path.join(NWCSAF_DIR, NWCSAF_NAME_FMT.format(valid_time))
    return path if os.path.exists(path) else None

def find_icon_analysis(exp, valid_time):
    path = os.path.join(ICON_DIRS[exp], ICON_NAME_FMT.format(valid_time))
    return path if os.path.exists(path) else None

def find_hpbl(exp, valid_time):
    path = os.path.join(HPBL_DIR, HPBL_NAME_FMT.format(exp=exp, time=valid_time))
    return path if os.path.exists(path) else None


# ==========================================================
# READERS
# ==========================================================

def load_icon_fieldlist(grib_file):
    return ekd.from_source("file", grib_file).to_fieldlist()

def extract_field(fields, param):
    sel = fields.sel({"parameter.variable": param})
    if len(sel) == 0:
        raise RuntimeError(f"{param} not found")
    xa = sel[0].to_xarray()
    # fieldextra output may name the variable differently from the GRIB shortName
    name = param if param in xa else list(xa.data_vars)[0]
    return xa["longitude"].values, xa["latitude"].values, xa[name].values

def read_hpbl(exp, valid_time):
    path = find_hpbl(exp, valid_time)
    if path is None:
        return None
    return extract_field(load_icon_fieldlist(path), "HPBL")


# ==========================================================
# NWCSAF -> ICON NEAREST NEIGHBOUR (tree cached, grid is static)
# ==========================================================

_NWC_TREE = None

def nwcsaf_tree(lon_nwc, lat_nwc):
    global _NWC_TREE
    if _NWC_TREE is None:
        pts = np.column_stack([lon_nwc.ravel(), lat_nwc.ravel()])
        good = np.isfinite(pts).all(axis=1)
        _NWC_TREE = (cKDTree(pts[good]), good)
    return _NWC_TREE

def regrid_to_icon(lon_nwc, lat_nwc, values, lon_icon, lat_icon):
    tree, good = nwcsaf_tree(lon_nwc, lat_nwc)
    dist, idx = tree.query(np.column_stack([lon_icon, lat_icon]), k=1)
    out = np.asarray(values).ravel()[good][idx].astype(float)
    out[dist > MAX_NN_DIST_DEG] = np.nan
    return out


# ==========================================================
# DISCOVER CASES
# ==========================================================

def collect_cases():
    cases = []
    t = SEARCH_START
    while t <= SEARCH_END:
        files = {
            "nwcsaf_file": find_nwcsaf_file(t),
            "icon801_file": find_icon_analysis("801", t),
            "icon802_file": find_icon_analysis("802", t),
        }
        if all(files.values()):
            cases.append({"valid_time": t, **files})
        t += timedelta(hours=1)
    return cases

def limit_cases(cases):
    """Apply FRAME_STRIDE then MAX_FRAMES."""
    selected = cases[::FRAME_STRIDE] if FRAME_STRIDE > 1 else cases
    if MAX_FRAMES is not None:
        selected = selected[:MAX_FRAMES]
    if len(selected) != len(cases):
        print(f"Limited to {len(selected)} of {len(cases)} cases "
              f"(MAX_FRAMES={MAX_FRAMES}, FRAME_STRIDE={FRAME_STRIDE})")
    return selected


# ==========================================================
# PANEL HELPERS
# ==========================================================

BIN_CMAP = ListedColormap(["#4575b4", "#f0f0f0", "#d73027"])   # -1 / 0 / +1
BIN_NORM = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], BIN_CMAP.N)

def style_map(ax, xlabels=True, ylabels=True):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    xlocs = list(range(MAP_EXTENT[0], MAP_EXTENT[1] + 1, 2))
    ylocs = list(range(MAP_EXTENT[2], MAP_EXTENT[3] + 1, 1))
    ax.gridlines(xlocs=xlocs, ylocs=ylocs, linewidth=0.3, alpha=0.5)

    ax.set_xticks(xlocs, crs=ccrs.PlateCarree())
    ax.set_yticks(ylocs, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(labelsize=TICK_SIZE)

    # only label the outer edges to keep the grid uncluttered
    if not xlabels:
        ax.set_xticklabels([])
    if not ylabels:
        ax.set_yticklabels([])

def placeholder(ax, title):
    ax.text(0.5, 0.5, "not available", ha="center", va="center",
            transform=ax.transAxes, fontsize=10, color="0.45")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=TITLE_SIZE)

def agreement_stats(diff):
    valid = np.isfinite(diff)
    n = valid.sum()
    if n == 0:
        return None
    d = diff[valid]
    return {"agree": (d == 0).sum() / n,
            "over":  (d == 1).sum() / n,
            "under": (d == -1).sum() / n}

def agreement_legend(ax, stats):
    """Colour key doubling as the stats readout, so neither is repeated."""
    if stats is None:
        return
    ax.legend(
        handles=[
            mpatches.Patch(color="#d73027",
                           label=f"too cloudy   {stats['over']:.0%}"),
            mpatches.Patch(color="#f0f0f0", ec="0.6",
                           label=f"agree        {stats['agree']:.0%}"),
            mpatches.Patch(color="#4575b4",
                           label=f"too clear    {stats['under']:.0%}"),
        ],
        loc="lower left", framealpha=0.85,
        handlelength=1.2, handleheight=0.8,
        borderpad=0.3, labelspacing=0.15, handletextpad=0.5,
        prop={"family": "monospace", "size": LEGEND_SIZE},
    )


# ==========================================================
# PROCESS ONE CASE
# ==========================================================

def process_case(case):
    valid_time = case["valid_time"]

    # --- NWCSAF cloud mask ---
    ds = xr.open_dataset(case["nwcsaf_file"])
    cma = ds["cma"].where(ds["cma"] != 255).values
    lon_nwc, lat_nwc = ds["longitude"].values, ds["latitude"].values
    ds.close()

    # --- ICON total cloud cover ---
    lon_801, lat_801, clct_801 = extract_field(
        load_icon_fieldlist(case["icon801_file"]), "CLCT")
    lon_802, lat_802, clct_802 = extract_field(
        load_icon_fieldlist(case["icon802_file"]), "CLCT")

    # --- binary agreement against the satellite mask ---
    sat_cloudy = regrid_to_icon(lon_nwc, lat_nwc, cma, lon_801, lat_801)

    diff_801 = (clct_801 > CLCT_CLOUDY_THRESHOLD).astype(float) - sat_cloudy
    diff_802 = (clct_802 > CLCT_CLOUDY_THRESHOLD).astype(float) - sat_cloudy

    stats_801 = agreement_stats(diff_801)
    stats_802 = agreement_stats(diff_802)

    # --- raw cloud cover difference between the experiments ---
    clct_diff = clct_801.astype(float) - clct_802.astype(float)
    clct_bias = np.nanmean(clct_diff)
    clct_mad = np.nanmean(np.abs(clct_diff))

    # --- HPBL ---
    hpbl_801 = read_hpbl("801", valid_time)
    hpbl_802 = read_hpbl("802", valid_time)

    hpbl_diff = None
    if hpbl_801 is not None and hpbl_802 is not None:
        if hpbl_801[2].size == hpbl_802[2].size:
            hpbl_diff = hpbl_801[2].astype(float) - hpbl_802[2].astype(float)

    # ------------------------------------------------------
    # PLOT
    # ------------------------------------------------------
    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(2, 3, figsize=(18, 8), layout="constrained",
                             subplot_kw={"projection": proj})
    (ax_d801, ax_d802, ax_dclct), (ax_h801, ax_h802, ax_hdiff) = axes

    fig.suptitle(f"{valid_time:%Y-%m-%d %H:%M UTC}", fontsize=13)

    mapped = {}   # ax -> (xlabels, ylabels)

    # --- row 1: agreement with NWCSAF, and the raw model difference ---
    ax_d801.scatter(lon_801, lat_801, c=diff_801, s=DOT_SIZE,
                    cmap=BIN_CMAP, norm=BIN_NORM, transform=proj)
    ax_d801.set_title("801 vs NWCSAF", fontsize=TITLE_SIZE)
    agreement_legend(ax_d801, stats_801)

    ax_d802.scatter(lon_802, lat_802, c=diff_802, s=DOT_SIZE,
                    cmap=BIN_CMAP, norm=BIN_NORM, transform=proj)
    ax_d802.set_title("802 vs NWCSAF", fontsize=TITLE_SIZE)
    agreement_legend(ax_d802, stats_802)

    sc_clct = ax_dclct.scatter(
        lon_801, lat_801, c=clct_diff, s=DOT_SIZE,
        cmap=CLCT_DIFF_CMAP, vmin=-CLCT_DIFF_LIM, vmax=CLCT_DIFF_LIM,
        transform=proj)
    ax_dclct.set_title("801 − 802 cloud cover", fontsize=TITLE_SIZE)

    mapped[ax_d801] = (False, True)
    mapped[ax_d802] = (False, False)
    mapped[ax_dclct] = (False, False)

    # --- row 2: HPBL and their difference ---
    sc_hpbl = None
    for ax, hpbl, exp, ylab in ((ax_h801, hpbl_801, "801", True),
                                (ax_h802, hpbl_802, "802", False)):
        if hpbl is None:
            placeholder(ax, f"{exp} boundary layer height")
            continue
        lon_h, lat_h, values = hpbl
        sc = ax.scatter(lon_h, lat_h, c=values, s=DOT_SIZE, cmap=HPBL_CMAP,
                        vmin=HPBL_VMIN, vmax=HPBL_VMAX, transform=proj)
        sc_hpbl = sc_hpbl or sc
        ax.set_title(f"{exp} boundary layer height", fontsize=TITLE_SIZE)
        mapped[ax] = (True, ylab)

    if hpbl_diff is None:
        placeholder(ax_hdiff, "801 − 802 boundary layer height")
        sc_hdiff = None
    else:
        sc_hdiff = ax_hdiff.scatter(
            hpbl_801[0], hpbl_801[1], c=hpbl_diff, s=DOT_SIZE,
            cmap=HPBL_DIFF_CMAP, vmin=-HPBL_DIFF_LIM, vmax=HPBL_DIFF_LIM,
            transform=proj)
        ax_hdiff.set_title("801 − 802 boundary layer height", fontsize=TITLE_SIZE)
        mapped[ax_hdiff] = (True, False)

    for ax, (xlab, ylab) in mapped.items():
        style_map(ax, xlabels=xlab, ylabels=ylab)

    # --- colourbars ---
    fig.colorbar(sc_clct, ax=ax_dclct, shrink=0.65, pad=0.02,
                label="cloud cover difference (%)")
    if sc_hpbl is not None:
        fig.colorbar(sc_hpbl, ax=[ax_h801, ax_h802], shrink=0.65, pad=0.02,
                    label="boundary layer height (m)")
    if sc_hdiff is not None:
        fig.colorbar(sc_hdiff, ax=ax_hdiff, shrink=0.70, pad=0.02,
                    label="height difference (m)")

    # --- numbers spelled out below the plots ---
    summary = (f"cloud cover 801 − 802:  mean difference {clct_bias:+.1f} % "
               f",  mean absolute difference {clct_mad:.1f}")
    if hpbl_diff is not None:
        summary += (f"          boundary layer height 801 − 802:  "
                    f"mean difference {np.nanmean(hpbl_diff):+.0f} m,  "
                    f"mean absolute difference {np.nanmean(np.abs(hpbl_diff)):.0f} m")
    fig.supxlabel(summary, fontsize=12)

    if SAVE_FIGURES:
        out_path = os.path.join(
            OUTPUT_DIR, f"CloudAgree_HPBL_{valid_time:%Y%m%d_%H%M}.png")
        fig.savefig(out_path, dpi=150)
        print("Saved:", out_path)

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":
    cases = collect_cases()
    if not cases:
        raise FileNotFoundError("No matching NWCSAF/801/802 files found")
    print(f"Found {len(cases)} cases "
          f"({cases[0]['valid_time']:%Y-%m-%d %H} .. {cases[-1]['valid_time']:%Y-%m-%d %H})")

    cases = limit_cases(cases)

    if PARALLEL and SAVE_FIGURES:
        import concurrent.futures
        with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
            list(pool.map(process_case, cases))
        n_processed = len(cases)
    else:
        for n_processed, case in enumerate(cases, start=1):
            print(f"[{n_processed}/{len(cases)}] {case['valid_time']:%Y-%m-%d %H:%M}")
            process_case(case)

    print(f"\nDone: {n_processed} frame(s) plotted.")

Found 143 cases (2025-10-04 01 .. 2025-10-10 00)
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0300.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0400.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0600.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0500.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0800.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0200.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0100.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_0700.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_1000.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_1400.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_1300.png
Saved: /users/jdelbeke/CloudAgreement_HPBL/CloudAgree_HPBL_20251004_1200.png
Saved: /users/jdelbeke/Clou